In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import torch.nn.functional as F
import os
os.listdir('/kaggle/working')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
import torch
import torch.nn as nn
import torch.nn.functional as F

# ----------------------------- Basic building blocks -----------------------------
class ConvBNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=None, groups=1, act=True):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        # Ensure group divides both in_ch and out_ch
        groups = min(groups, in_ch)
        if out_ch % groups != 0:
            groups = 1  # fallback
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.GELU() if act else nn.Identity()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class DepthwiseSeparable(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1):
        super().__init__()
        self.dw = ConvBNAct(in_ch, in_ch, kernel_size=kernel_size, stride=stride, groups=in_ch)
        self.pw = ConvBNAct(in_ch, out_ch, kernel_size=1, stride=1)

    def forward(self, x):
        return self.pw(self.dw(x))


# ----------------------------- Multi-scale stem ----------------------------------
class MultiScaleStem(nn.Module):
    def __init__(self, in_ch=3, out_ch=32):  # was 64
        super().__init__()
        half = out_ch // 2
        self.conv3 = ConvBNAct(in_ch, half, kernel_size=3)
        self.dw5 = ConvBNAct(in_ch, half, kernel_size=5, groups=min(in_ch, 3))
        self.conv1 = ConvBNAct(in_ch, half, kernel_size=1)
        self.project = ConvBNAct(half * 3, out_ch, kernel_size=1)

    def forward(self, x):
        a = self.conv3(x)
        b = self.dw5(x)
        c = self.conv1(x)
        cat = torch.cat([a, b, c], dim=1)
        return self.project(cat)


# ----------------------------- Local, Global, Channel Encoders ------------------
class LocalEncoder(nn.Module):
    def __init__(self, in_ch, out_ch, expansion=1.0):
        super().__init__()
        mid = int(in_ch * expansion)
        self.block = nn.Sequential(
            DepthwiseSeparable(in_ch, mid, kernel_size=3),
            ConvBNAct(mid, out_ch, kernel_size=1)
        )

    def forward(self, x):
        return self.block(x)


class GlobalEncoder(nn.Module):
    def __init__(self, dim, pool_size=7, heads=4):
        super().__init__()
        self.pool_size = pool_size
        self.heads = heads
        self.scale = (dim // heads) ** -0.5 if dim >= heads else 1.0
        self.to_q = nn.Conv2d(dim, dim, 1, bias=False)
        self.to_k = nn.Conv2d(dim, dim, 1, bias=False)
        self.to_v = nn.Conv2d(dim, dim, 1, bias=False)
        self.out = nn.Conv2d(dim, dim, 1)

    def forward(self, x):
        b, c, h, w = x.shape
        pooled = F.adaptive_avg_pool2d(x, (self.pool_size, self.pool_size))
        q = self.to_q(pooled).flatten(2)
        k = self.to_k(pooled).flatten(2)
        v = self.to_v(pooled).flatten(2)
        attn = torch.bmm(q.transpose(1, 2), k) * self.scale
        attn = attn.softmax(dim=-1)
        out = torch.bmm(v, attn.transpose(1, 2)).view(b, c, self.pool_size, self.pool_size)
        out = F.interpolate(self.out(out), size=(h, w), mode='bilinear', align_corners=False)
        return out


class ChannelEncoder(nn.Module):
    def __init__(self, channels, conv_kernel=3):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        padding = conv_kernel // 2
        self.conv1d = nn.Conv1d(1, 1, kernel_size=conv_kernel, padding=padding, bias=False)
        self.act = nn.Sigmoid()
        self.expand = nn.Sequential(
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):
        z = self.avg(x).squeeze(-1).transpose(1, 2)
        z = self.conv1d(z)
        z = self.act(z).transpose(1, 2).unsqueeze(-1)
        out = x * z
        return self.expand(out)


# ----------------------------- Cross Interaction & Fusion -----------------------
class CrossInteraction(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.modulator = nn.Sequential(
            nn.Conv2d(channels * 3, channels, 1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(channels, channels, 1),
            nn.Sigmoid()
        )

    def forward(self, local, global_f, channel_f):
        cat = torch.cat([local, global_f, channel_f], dim=1)
        mod = self.modulator(cat)
        return local * mod, global_f * mod, channel_f * mod


class CLGFusionBlock(nn.Module):
    def __init__(self, channels, pool_size=7, expansion=1.0):
        super().__init__()
        self.local = LocalEncoder(channels, channels, expansion)
        self.global_enc = GlobalEncoder(channels, pool_size)
        self.channel_enc = ChannelEncoder(channels)
        self.cross = CrossInteraction(channels)
        self.fusion_logits = nn.Parameter(torch.tensor([1.0, 1.0, 1.0]))
        self.norm = nn.BatchNorm2d(channels)
        self.project = ConvBNAct(channels, channels, 1)

    def forward(self, x):
        local_f = self.local(x)
        global_f = self.global_enc(x)
        channel_f = self.channel_enc(x)
        local_m, global_m, channel_m = self.cross(local_f, global_f, channel_f)
        w = F.softmax(self.fusion_logits, dim=0)
        fused = w[0] * local_m + w[1] * global_m + w[2] * channel_m
        fused = self.project(self.norm(fused))
        return F.gelu(fused + x)


class CLoGNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2, widths=[64, 96, 160], blocks=[1, 2, 2], pool_size=7):
        super().__init__()
        self.stem = MultiScaleStem(in_channels, widths[0])
        layers = []
        in_ch = widths[0]
        for i, (w, n) in enumerate(zip(widths, blocks)):
            if w != in_ch:
                layers.append(ConvBNAct(in_ch, w, 1))
                in_ch = w
            for _ in range(n):
                layers.append(CLGFusionBlock(in_ch, pool_size))
            if i != len(widths) - 1:
                layers.append(nn.MaxPool2d(2))
        self.body = nn.Sequential(*layers)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_ch, in_ch // 2),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(in_ch // 2, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.body(x)
        x = self.head(x)
        return x


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)



cuda
cuda


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# =========================
# Transform (NO augmentation, NO resize)
# =========================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# =========================
# Dataset
# =========================

base_path = "/kaggle/input/datasets/sadnasingh/deepfake-ds/dataset_1"
train_dataset = ImageFolder(root=f"{base_path}/train", transform=transform)
val_dataset   = ImageFolder(root=f"{base_path}/val",   transform=transform)
test_dataset  = ImageFolder(root=f"{base_path}/test",  transform=transform)

# =========================
# Optimized DataLoaders for T4
# =========================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,          # increased from 8
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Datasets and DataLoaders ready.")

cuda


Datasets and DataLoaders ready.


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import copy

# ==============================
# Device
# ==============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ==============================
# Model
# ==============================
model = CLoGNet(
    in_channels=3,
    num_classes=2,
    widths=[32, 48, 80],
    blocks=[1, 1, 1],
    pool_size=5
).to(device)

print(f"Model Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.3f}M")

# ==============================
# Loss
# ==============================
criterion = nn.CrossEntropyLoss()

# ==============================
# Optimizer
# ==============================
optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# ==============================
# Scheduler
# ==============================
num_epochs = 50
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

# ==============================
# Mixed Precision
# ==============================
scaler = GradScaler()

# ==============================
# Checkpoint Setup
# ==============================
checkpoint_path = "/kaggle/input/notebooks/sadnasingh/clognet-optimized/checkpoint.pth"
best_model_path = "/kaggle/input/notebooks/sadnasingh/clognet-optimized/best_model.pth"

start_epoch = 0
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

best_val_loss = float("inf")
patience = 8
early_stop_counter = 0

# ==============================
# Resume if checkpoint exists
# ==============================
if os.path.exists(checkpoint_path):
    print("Loading checkpoint...")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state"])
    optimizer.load_state_dict(checkpoint["optimizer_state"])
    scheduler.load_state_dict(checkpoint["scheduler_state"])
    scaler.load_state_dict(checkpoint["scaler_state"])

    start_epoch = checkpoint["epoch"] + 1
    train_losses = checkpoint["train_losses"]
    val_losses = checkpoint["val_losses"]
    train_accuracies = checkpoint["train_accuracies"]
    val_accuracies = checkpoint["val_accuracies"]
    best_val_loss = checkpoint["best_val_loss"]

    print(f"Resuming from epoch {start_epoch}")
else:
    print("Training from scratch.")

# ==============================
# Training Loop
# ==============================
checkpoint_path = "/kaggle/working/checkpoint.pth"
best_model_path = "/kaggle/working/best_model.pth"
for epoch in range(start_epoch, num_epochs):

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print("-" * 40)

    # ==========================
    # TRAIN
    # ==========================
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    for images, labels in tqdm(train_loader):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()

        # Gradient clipping
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (preds == labels).sum().item()

    avg_train_loss = train_loss / len(train_loader)
    train_acc = 100 * train_correct / train_total

    train_losses.append(avg_train_loss)
    train_accuracies.append(train_acc)

    # ==========================
    # VALIDATION
    # ==========================
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (preds == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * val_correct / val_total

    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)

    print(f"Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.2f}%")
    print(f"Val   Loss: {avg_val_loss:.4f} | Acc: {val_acc:.2f}%")

    # ==========================
    # Scheduler Step
    # ==========================
    scheduler.step()

    # ==========================
    # Save Regular Checkpoint
    # ==========================
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "train_losses": train_losses,
        "val_losses": val_losses,
        "train_accuracies": train_accuracies,
        "val_accuracies": val_accuracies,
        "best_val_loss": best_val_loss
    }, checkpoint_path)

    # ==========================
    # Save Best Model
    # ==========================
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stop_counter = 0

        torch.save(model.state_dict(), best_model_path)
        print("Validation improved. Best model saved.")

    else:
        early_stop_counter += 1
        print(f"No improvement ({early_stop_counter}/{patience})")

        if early_stop_counter >= patience:
            print("Early stopping triggered.")
            break

print("\nTraining Complete.")
print(f"Best Validation Loss: {best_val_loss:.4f}")

Device: cuda


/tmp/ipykernel_24/2688418780.py:52: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Model Parameters: 0.133M
Loading checkpoint...
Resuming from epoch 18

Epoch [19/50]
----------------------------------------


  0%|          | 0/2660 [00:00<?, ?it/s]

/tmp/ipykernel_24/2688418780.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


...

 41%|████      | 1088/2660 [26:51<38:55,  1.49s/it]

 41%|████      | 1089/2660 [26:52<38:58,  1.49s/it]

 41%|████      | 1090/2660 [26:54<38:57,  1.49s/it]

 41%|████      | 1091/2660 [26:55<38:55,  1.49s/it]

 41%|████      | 1092/2660 [26:57<38:55,  1.49s/it]

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_curve, auc
)

model.eval()

y_true = []
y_pred = []
y_probs = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)

        outputs = model(images)                 # logits
        probs = torch.softmax(outputs, dim=1)   # probabilities
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())
        y_probs.extend(probs.cpu().numpy())

# Convert to numpy
y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_probs = np.array(y_probs)

acc = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Test Accuracy :", acc)
print("Precision     :", precision)
print("Recall        :", recall)
print("F1 Score      :", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()


fpr, tpr, _ = roc_curve(y_true, y_probs[:, 1])
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.grid()
plt.show()

print("ROC–AUC Score :", roc_auc)
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.legend()
plt.title("Loss Curve")

plt.subplot(1,2,2)
plt.plot(train_accuracies, label='Train Acc')
plt.plot(val_accuracies, label='Val Acc')
plt.legend()
plt.title("Accuracy Curve")

plt.show()